# Notebook to Combine Serializations and Labels to Splits


In [ ]:
import pandas as pd
import pickle
import json
from utils import process_chexpert_labels, convert_multiclass_to_binary_labels
import numpy as np

In [ ]:
# Definitions
tasks = ["guo_los", "guo_readmission", "guo_icu",
         "new_pancan", "new_celiac", "new_lupus", "new_acutemi", "new_hypertension", "new_hyperlipidemia",
         "lab_thrombocytopenia", "lab_hyperkalemia", "lab_hypoglycemia", "lab_hyponatremia", "lab_anemia"]
chexpert_tasks = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Lesion", "Lung Opacity",
                  "Edema", "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
                  "Pleural Other", "Fracture", "Support Devices"]
all_tasks = tasks + [f"chexpert_{task}" for task in chexpert_tasks]
benchmark_folder = "/home/sthe14/ehrshot-benchmark/EHRSHOT_ASSETS/benchmark"
splits_json_path = "/home/sthe14/ehrshot-benchmark/EHRSHOT_ASSETS/splits/splits.json"

output_csv_path = f"{benchmark_folder}/ehrshot_splits_to_serializations.csv"

In [ ]:
# Read serializations and labels
serializations_and_instructions_path = f"{benchmark_folder}/serializations_instructions_20.pkl"
# Columns: patient_id,prediction_time,label_type,value,task
labels_path = f"{benchmark_folder}/all_labels_tasks.csv"

with open(serializations_and_instructions_path, 'rb') as f:
    serializations_and_instructions = pickle.load(f)
# Read csv as pandas dataframe, header exists
labels = pd.read_csv(labels_path)

# Debug: Extend serializations_and instructions to 1161412 entries by adding dummy entries
num_dummies = 1161412 - len(serializations_and_instructions)
if num_dummies > 0:
    print(f"Extending serializations_and_instructions with {num_dummies} dummy entries.")
    for i in range(num_dummies):
        serializations_and_instructions.append((f"dummy_input_{i}", f"dummy_output_{i}"))

# Cache location of patient id, task, and prediction time in all_labels_tasks.csv to find index of serialization and instruction
labels_row_dict = {(int(row['patient_id']), row['task'], row['prediction_time']): int(i) for i, row in labels.iterrows()}
assert max(labels_row_dict.values()) == len(serializations_and_instructions) - 1, "Labels does not match serializations / instructions"

# Read splits json as dictionary "val", "test", "train" and sets of integer patient ids
with open(splits_json_path, 'r') as f:
    splits = json.load(f)


In [ ]:
# Create splits per task
all_entries = []

def append_entry(task, split_name, shot_size, fold, patient_id, prediction_time, label_type, label_value):
    prediction_time = prediction_time.replace("T", " ")
    # Get index of serialization and instruction
    task_lookup = task if not task.startswith("chexpert_") else "chexpert"
    idx = labels_row_dict.get((patient_id, task_lookup, prediction_time))
    assert idx is not None, f"Patient ID {patient_id}, task {task}, label time {prediction_time} not in labels"
    all_entries.append((task, split_name, shot_size, fold, patient_id, prediction_time, label_type, label_value, idx))
    
def process_label(task, label_value):
    # Checked in ehrshot code; task always framed as binary task
    if task.startswith("chexpert"):
        processed_label = process_chexpert_labels([label_value])[0][chexpert_tasks.index(task.replace("chexpert_", ""))]
    elif task.startswith("lab"):
        # Treat all non normal as abnormal
         processed_label = convert_multiclass_to_binary_labels(np.array([label_value]), threshold=1)[0]
    else:
        processed_label = label_value
    return bool(processed_label)

for task in all_tasks:
    # if not task.startswith("chexpert"):
    #     continue  # Skip chexpert tasks for now, they are handled separately
    
    if task.startswith("chexpert"):
        task_splits_path = f"{benchmark_folder}/chexpert/all_shots_data.json"
        labeled_patients_path = f"{benchmark_folder}/chexpert/labeled_patients.csv"
    else:
        task_splits_path = f"{benchmark_folder}/{task}/all_shots_data.json"
        labeled_patients_path = f"{benchmark_folder}/{task}/labeled_patients.csv"
        
    # Read in task specific labels
    labeled_patients = pd.read_csv(labeled_patients_path)
    labeled_patients_dict = {(int(row['patient_id']), row['prediction_time'].replace("T", " ")): (row['label_type'], row['value']) for i, row in labeled_patients.iterrows()}

    def get_label_type_value(patient_id, prediction_time):
        (label_type, label_value) = labeled_patients_dict.get((int(patient_id), prediction_time.replace("T", " ")), (None, None))
        if label_type is None or label_value is None:
            raise ValueError(f"Label for patient ID {patient_id} and time {prediction_time} not found.")
        return label_type, label_value
    
    
    with open(task_splits_path, 'r') as f:
        if task.startswith("chexpert"):
            task_splits = json.load(f)[task.replace("chexpert_", "")]
        else:
            task_splits = json.load(f)[task]
        for shot_size in task_splits:
            print(f"Processing task {task} with shot size {shot_size}")
            for fold in task_splits[shot_size]:
                for i, id in enumerate(task_splits[shot_size][fold]['patient_ids_train_k']):
                    assert int(id) in splits['train'], f"Patient ID {id} not in train split for task {task}"
                    (label_type, label_value) = get_label_type_value(int(id), task_splits[shot_size][fold]['label_times_train_k'][i])
                    append_entry(task, 'train', shot_size, fold, int(id), task_splits[shot_size][fold]['label_times_train_k'][i], label_type, process_label(task, label_value))
                
                for i, id in enumerate(task_splits[shot_size][fold]['patient_ids_val_k']):
                    (label_type, label_value) = get_label_type_value(int(id), task_splits[shot_size][fold]['label_times_val_k'][i])
                    assert int(id) in splits['val'], f"Patient ID {id} not in val split for task {task}"
                    append_entry(task, 'val', shot_size, fold, int(id), task_splits[shot_size][fold]['label_times_val_k'][i], label_type, process_label(task, label_value))

    # Now store all test examples from labels with respective task and patient_id
    test_task = task if not task.startswith("chexpert") else "chexpert"
    test_rows = labels[(labels['task'] == test_task) & (labels['patient_id'].isin(set(splits['test'])))]
    for _, row in test_rows.iterrows():
        (label_type, label_value) = get_label_type_value(row['patient_id'], row['prediction_time'])
        append_entry(task, 'test', 0, 0, int(row['patient_id']), row['prediction_time'], label_type, process_label(task, label_value))

# Do final check that ids belong to original splits
for entry in all_entries:
    task, split_name, shot_size, fold, patient_id, prediction_time, label_type, label_value, idx = entry
    assert patient_id in splits[split_name], f"Patient ID {patient_id} not in {split_name} split for task {task}"


# Write out as pandas dataframe
df = pd.DataFrame(all_entries, columns=['task', 'split_name', 'shot_size', 'fold', 'patient_id', 'prediction_time', 'label_type', 'label_value', 'serialization_idx'])
df.to_csv(output_csv_path, index=False) 

In [ ]:
set(df['label_value'])  # Check that all tasks are present